we train a Random Forest classifier on the same 5 input variables
used in the fuzzy logic system

this model will be used in the streamlit app as a comparison against Mamdani and Sugeno

this does NOT replace the fuzzy system, it serves as a benchmark

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, confusion_matrix
import joblib
import os

In [ ]:
df = pd.read_csv("../data/gtd_processed.csv")

prop_map = {1: 3, 2: 2, 3: 1, 4: 0}
df["prop_inverted"] = df["propextent"].map(prop_map)

X = df[["nkill", "nwound", "prop_inverted", "attack_encoded", "weapon_encoded"]]
y = df["severity_index"]

print(f"Total samples : {len(df):,}")
print(f"Features      : {list(X.columns)}")
print(f"Class distribution:")
print(y.value_counts())

In [ ]:
# train test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size : {len(X_train):,}")
print(f"Test size  : {len(X_test):,}")

In [ ]:
# train random forest

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

In [ ]:
# eval

y_pred = rf.predict(X_test)
order  = ["Low", "Medium", "High", "Critical"]

acc = accuracy_score(y_test, y_pred)
print(f"Random Forest Accuracy: {acc:.4f} ({acc*100:.2f}%)\n")
print(classification_report(y_test, y_pred, labels=order, target_names=order))

In [ ]:
# confusion matrix

cm = confusion_matrix(y_test, y_pred, labels=order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=order)

fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap="Greens", colorbar=False)
plt.title("Random Forest Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# feature importance

feature_names = ["nkill", "nwound", "prop_inverted", "attack_encoded", "weapon_encoded"]
importances = rf.feature_importances_

plt.figure(figsize=(8, 4))
plt.barh(feature_names, importances, color="#27ae60")
plt.title("Feature Importance: Random Forest")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
joblib.dump(rf, "../app/model/rf_model.pkl")
print("saved to app/model/rf_model.pkl")

## Summary

- Random Forest classifier trained on 5 input variables with 80/20 train-test split
- Training samples: 139,532 
- Test samples: 34,883
- Random Forest achieved 99.95% accuracy on the test set,
  significantly higher than both Mamdani and Sugeno at 76.15%
- Precision, recall, and F1-score all near 1.00 for all 4 severity classes
- Feature importance shows nkill (55%) and nwound (39%) are the dominant features,
  while attack_encoded, weapon_encoded, and prop_inverted have minimal impact
- This makes sense because our ground truth severity_index was defined
  primarily based on casualty numbers
- Model saved to app/model/rf_model.pkl for use in the Streamlit app
- Random Forest does NOT replace the fuzzy system, fuzzy remains the main system, RF serves as a benchmark comparison